<a href="https://colab.research.google.com/github/amirgroup-codes/ProGenMech/blob/main/ProGenMech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="left">
  <img src="https://raw.githubusercontent.com/amirgroup-codes/ProGenMech/main/ProGenMech_Logo_Glow.svg"
       alt="ProGenMech"
       width="60%">
</p>

# ProGenMech: Circuit Tracing in Autoregressive Protein Language Models

ProGenMech is a framework for discovering computational circuits in the autoregressive protein language model ProGen3 using cross-layer transcoders (CLTs). This Colab notebook is designed to produce the files required for our [website](https://protmech.github.io/):
1. `activation_indices.json`
2. `generation.fasta`
3. `top_activations.json`
4. `virtual_weights.json`

A link to the paper can be found [here](https://arxiv.org/abs/2606.16044). We additionally provide our [code](https://github.com/amirgroup-codes/ProGenMech), [models](https://huggingface.co/darintsui/ProGenMechModels), and [data](https://huggingface.co/datasets/darintsui/ProGenMechData).

Unlike ProtoMech (ESM2), ProGenMech uses ProGen3 with specialized dependencies (`external/progen3`, `megablocks`, `triton`, `flash-attn`). **Use a T4 GPU or better.** On Colab T4 (sm_75), ProGen3 is automatically reloaded with `moe_implementation='eager'` (no megablocks/Triton MoE) plus PyTorch RMSNorm and fallback attention; A100/L4 are faster.

This notebook has **two independent workflows** — run either one:
- **CLM circuit discovery** — autoregressive generation from a prompt sequence.
- **Zero-shot circuit discovery** — fitness prediction from a regression CSV + sequence scoring.

---

In [20]:
# @title 0. Check GPU status and install dependencies
import os
import sys
import argparse
import importlib
import pty
import subprocess
import shutil
import ipywidgets as widgets

# 1. Clean up stale local directories (Leave pip cache intact so we don't redownload from the web)
print("Clearing old local compilation builds...")
!rm -rf /usr/local/lib/python3.12/dist-packages/flash_attn*
!rm -rf /usr/local/lib/python3.12/dist-packages/megablocks*
!rm -rf /usr/local/lib/python3.12/dist-packages/grouped_gemm*

# 2. Synchronize basic ecosystem prerequisites (Upgraded packaging resolves bigquery errors early)
print("Setting up baseline environment tools...")
!pip install -q "packaging>=24.2.0" wheel setuptools meson-python ninja patchelf "pillow<12.0" "fsspec[http]==2025.3.0" "numpy>=1.22,<2.1.0"

# 3. Align PyTorch ecosystem components (Only installs once if not present)
print("Verifying synchronized PyTorch layout...")
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124

import torch
if torch.cuda.is_available():
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    print(f"GPU Active: {torch.cuda.get_device_name(0)} (sm_{cc_major}{cc_minor})")
else:
    print("WARNING: No GPU detected.")

if hasattr(torch.serialization, 'add_safe_globals'):
    torch.serialization.add_safe_globals([argparse.Namespace])

# 4. Install Core Framework Ecosystem
print("Syncing model framework libraries...")
!pip install -q huggingface_hub pytorch-lightning pandas scipy einops "transformers>=4.42,<4.49" tqdm ipywidgets torchmetrics

# 5. Native Compilation (Crucial Speedup: --no-deps prevents redownloading 900MB PyTorch)
print("Compiling MoE layers natively from source (MegaBlocks / Grouped-Gemm)...")
# Changed --force-reinstall to --upgrade --no-deps so it won't touch the existing torch layout
!pip install --no-binary megablocks,grouped-gemm --no-build-isolation --upgrade --no-deps -q "megablocks[gg]==0.7.0" grouped-gemm stanford-stk

# 6. Safe Flash Attention Fallback Evaluation
try:
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    abi = "TRUE" if torch.compiled_with_cxx11_abi() else "FALSE"
    wheel = (
        f"https://github.com/Dao-AILab/flash-attention/releases/download/"
        f"v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch{torch_mm}cxx11abi{abi}-"
        f"{py_tag}-{py_tag}-linux_x86_64.whl"
    )
    rc = os.system(f"pip install -q --no-deps {wheel}")
except Exception:
    pass

# 7. Verification Verification Post-install
print("\n--- Verification Status ---")
for module in ["megablocks", "triton", "pytorch_lightning", "torchvision"]:
    try:
        importlib.reload(importlib.import_module(module))
        print(f"[-] {module}: READY")
    except ImportError:
        print(f"[X] {module}: Missing")

print("Ecosystem initialization sequence finalized.")

Clearing old local compilation builds...
Setting up baseline environment tools...
Verifying synchronized PyTorch layout...
GPU Active: Tesla T4 (sm_75)
Syncing model framework libraries...
Compiling MoE layers natively from source (MegaBlocks / Grouped-Gemm)...

--- Verification Status ---
[-] megablocks: READY
[-] triton: READY
[-] pytorch_lightning: READY
[-] torchvision: READY
Ecosystem initialization sequence finalized.


In [ ]:
# @title 1. Clone ProGenMech and download models and data
from huggingface_hub import hf_hub_download, snapshot_download

REPO_URL = "https://github.com/amirgroup-codes/ProGenMech.git"
BRANCH = "main"
TARGET_DIR = "/content/ProGenMech"

MODEL_REPO = "darintsui/ProGenMechModels"
DATA_REPO = "darintsui/ProGenMechData"
PROGEN3_MODEL = "Profluent-Bio/progen3-112m"
clt_filename = "ProGen3_CLT_L10_D4608/checkpoints/last.ckpt"
activations_filename = "top10_activations.pt"

# 1. Clean up or check for existing directory
if os.path.exists(TARGET_DIR):
    print(f"Directory {TARGET_DIR} already exists. Preparing for a fresh clone...")
    shutil.rmtree(TARGET_DIR)

# 2. Clone the repository (Public URL)
print(f"Cloning {REPO_URL}...")
result = os.system(f"git clone -q -b {BRANCH} {REPO_URL} {TARGET_DIR}")

if result == 0:
    print(f"Successfully cloned ProGenMech into {TARGET_DIR}")
else:
    raise RuntimeError("Failed to clone repository.")

# 3. Add to sys.path so imports work immediately
if TARGET_DIR not in sys.path:
    sys.path.append(TARGET_DIR)
PROGEN3_SRC = os.path.join(TARGET_DIR, "external", "progen3", "src")
if PROGEN3_SRC not in sys.path:
    sys.path.append(PROGEN3_SRC)

# 4. Install vendored ProGen3 package
PROGEN3_ROOT = os.path.join(TARGET_DIR, "external", "progen3")
if os.path.isdir(PROGEN3_ROOT):
    !pip install -q -e {PROGEN3_ROOT}
else:
    raise FileNotFoundError(f"Missing external/progen3 at {PROGEN3_ROOT}")

# 5. Download CLT checkpoint and reference activations
print("Downloading ProGenMech models and data...")
MODELS_DIR = os.path.join(TARGET_DIR, "models")
VISUALIZATION_DIR = os.path.join(TARGET_DIR, "visualization")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(VISUALIZATION_DIR, exist_ok=True)

hf_hub_download(
    repo_id=MODEL_REPO,
    filename=clt_filename,
    repo_type="model",
    local_dir=MODELS_DIR,
)
hf_hub_download(
    repo_id=DATA_REPO,
    filename=activations_filename,
    repo_type="dataset",
    local_dir=VISUALIZATION_DIR,
)

clt_checkpoint = os.path.join(MODELS_DIR, clt_filename)
activations_pt = os.path.join(VISUALIZATION_DIR, activations_filename)

os.environ["CLT_CHECKPOINT"] = clt_checkpoint
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PROGEN3_MODEL"] = PROGEN3_MODEL

if not os.path.exists(clt_checkpoint):
    raise ValueError(f"CLT not found at {clt_checkpoint}, check download")
if not os.path.exists(activations_pt):
    raise ValueError(f"{activations_pt} not found, check download")

# 6. Prefetch ProGen3 base weights (required when loading the CLT checkpoint)
print(
    "\nPrefetching ProGen3 base weights.\n"
    "ProGen3 is released under a non-commercial license — review the terms at\n"
    f"https://huggingface.co/{PROGEN3_MODEL} before use."
)
try:
    progen3_weights_dir = snapshot_download(repo_id=PROGEN3_MODEL)
    print(f"ProGen3 weights cached at: {progen3_weights_dir}")
except Exception as e:
    print(
        "Could not download ProGen3 automatically. If the model page requires access approval,\n"
        f"visit https://huggingface.co/{PROGEN3_MODEL}, accept the license, then run:\n"
        "  from huggingface_hub import login; login()\n"
        f"Error: {e}"
    )
    raise

TOY_REGRESSION_CSV = os.path.join(TARGET_DIR, "colab_examples", "toy_regression.csv")

REPO_ROOT = TARGET_DIR
SCRIPT_PATH = os.path.join(REPO_ROOT, "visualization", "auto_discover_circuit_website.py")
EDGE_SCRIPT = os.path.join(REPO_ROOT, "visualization", "get_edge_weights.py")


def run_realtime(cmd):
    master, slave = pty.openpty()
    p = subprocess.Popen(cmd, stdout=slave, stderr=slave, close_fds=True)
    os.close(slave)
    try:
        while True:
            try:
                data = os.read(master, 1024).decode()
                if not data:
                    break
                sys.stdout.write(data)
                sys.stdout.flush()
            except OSError:
                break
    except Exception:
        pass
    p.wait()
    os.close(master)
    return p.returncode


def collect_weight_folders(base_dir):
    folders = []
    if os.path.isdir(base_dir):
        for name in sorted(os.listdir(base_dir)):
            path = os.path.join(base_dir, name)
            if os.path.isdir(path) and name.startswith("seq"):
                folders.append(path)
    if (
        os.path.exists(os.path.join(base_dir, "generation.fasta"))
        or os.path.exists(os.path.join(base_dir, "seq.txt"))
    ):
        folders.insert(0, base_dir)
    seen = set()
    unique = []
    for path in folders:
        if path not in seen:
            seen.add(path)
            unique.append(path)
    return unique


def run_edge_weights(base_dir, label=""):
    print(f"\n[Edge weights] Computing virtual_weights.json{' for ' + label if label else ''}...")
    failed = False
    for folder_path in collect_weight_folders(base_dir):
        folder_name = os.path.basename(folder_path)
        print(f"\n   Processing {folder_name}...")
        weight_cmd = [
            sys.executable, EDGE_SCRIPT,
            "--base_folder", folder_path,
            "--clt_ckpt", clt_checkpoint,
        ]
        if run_realtime(weight_cmd) != 0:
            print(f"Failed to compute weights for {folder_name}")
            failed = True
    return not failed


print(f"\nCLT checkpoint: {clt_checkpoint}")
print(f"Reference activations: {activations_pt}")
print(f"Toy regression CSV: {TOY_REGRESSION_CSV}")
print("ProGenMech setup complete.")

---
## CLM circuit discovery

Use this section to discover a **generation circuit**: ProGen3 autoregressively fills in the next amino acid(s) from your prompt, then ProGenMech finds the CLT circuit and writes website-ready files.

**Workflow**
1. Configure your CLM prompt in the next cell (default: kinase example, generate **1** amino acid).
2. Run discovery + edge weights in the cell after that.

**Outputs** (in your chosen folder): `experiment.json`, `generation.fasta`, `top_activations.json`, `activation_indices.json`, `virtual_weights.json`, and related files.

In [ ]:
# @title CLM — Configure prompt
# @markdown **CLM settings**

# @markdown Prompt sequence (ProGen3 continues from the end):
clm_sequence = "MVKQVDFAEVKLSEKFLGAGSGGAVRKATFQNQEIAVKIFDFLEETIKKNAEREITHLSEIDHENVIRVIGRASNGKKDYLLMEYLEEGSLHNYLYGDDKWEYTVEQAVRWALQCAKALAYLHSLDRPIVHR"  # @param {type:"string"}
# @markdown Number of amino acids to generate (default: 1):
n_generated = 1  # @param {type:"integer"}
# @markdown 1-indexed generation start position (leave 0 to append at end):
clm_pos = 0  # @param {type:"integer"}

# @markdown Output folder name:
clm_output_dir = "clm_circuit"  # @param {type:"string"}
# @markdown Where to save results:
external_path = "/content/experiments"  # @param {type:"string"}
# @markdown Entry name for output JSON:
clm_entry_name = "experiment"  # @param {type:"string"}

if not clm_sequence.strip():
    raise ValueError("CLM prompt sequence cannot be empty.")

clm_full_output_dir = os.path.join(external_path, clm_output_dir)
os.makedirs(clm_full_output_dir, exist_ok=True)

print(f"CLM prompt length: {len(clm_sequence.strip())}")
print(f"Will generate {n_generated} amino acid(s)")
print(f"Output directory: {clm_full_output_dir}")

In [ ]:
# @title CLM — Run discovery and edge weights
# @markdown Runs circuit discovery for the CLM prompt, then computes `virtual_weights.json` (may take a while on Colab).

cmd = [
    sys.executable, SCRIPT_PATH,
    "--type", "CLM",
    "--output_dir", clm_full_output_dir,
    "--entry_name", clm_entry_name,
    "--seq", clm_sequence.strip(),
    "--n_generated", str(n_generated),
    "--max_nodes", "1000",
    "--step_size", "32",
]
if clm_pos and clm_pos > 0:
    cmd += ["--pos", str(clm_pos)]

print("Starting CLM discovery...")
print(f"   Output: {clm_full_output_dir}/{clm_entry_name}.json\n")
exit_code = run_realtime(cmd)

if exit_code != 0:
    print("\nCLM discovery failed.")
else:
    ok = run_edge_weights(clm_full_output_dir, label="CLM")
    print("\n==========")
    print(f"CLM results saved to: {clm_full_output_dir}")
    print("==========")
    print("Files generated:")
    for f in sorted(os.listdir(clm_full_output_dir)):
        suffix = "/" if os.path.isdir(os.path.join(clm_full_output_dir, f)) else ""
        print(f" - {f}{suffix}")
    if not ok:
        print("\nEdge-weight computation failed. See logs above.")

---
## Zero-shot circuit discovery

Use this section to discover a **fitness prediction circuit** from a regression CSV, then score/visualize specific sequences on the website.

**Workflow**
1. Configure CSV input in the next cell — upload your own file (like ProtoMech) or use the example CSV.
2. Add sequences to score (first = wildtype/reference).
3. Run discovery + edge weights.

### **Input CSV format (regression)**
Your CSV must have two columns (not case-sensitive):
* **Sequence:** `sequence` or `mutated_sequence` (all rows must be the same length).
* **Score:** `score`, `DMS_score`, or `target` (continuous numeric values).

**Example:**
```csv
mutant,mutated_sequence,DMS_score
K3R,MSR...LYK,3.74
K3Q,MSQ...LYK,3.75
K3E,MSE...LYK,3.67
```

The example CSV (`colab_examples/toy_regression.csv`) contains the first 10 entries of the GRB2 DMS assay.

In [ ]:
# @title Zero-shot — Configure CSV
# @markdown **Zero-shot settings**

# @markdown Check to upload your own regression CSV:
upload_csv = False  # @param {type:"boolean"}
# @markdown Use example CSV instead:
use_example_csv = True  # @param {type:"boolean"}

# @markdown Output folder name:
zs_output_dir = "zero_shot_circuit"  # @param {type:"string"}
# @markdown Where to save results:
zs_external_path = "/content/experiments"  # @param {type:"string"}
# @markdown Entry name for output JSON (used with toy CSV):
zs_entry_name = "experiment"  # @param {type:"string"}

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    files = None

ZERO_SHOT_CSV_PATH = None
ZERO_SHOT_ENTRY_NAME = zs_entry_name

if upload_csv:
    if not IN_COLAB:
        raise RuntimeError("CSV upload requires Google Colab.")
    print("Please upload your regression CSV file...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Upload cancelled.")
    csv_name = list(uploaded.keys())[0]
    ZERO_SHOT_CSV_PATH = os.path.join(os.getcwd(), csv_name)
    ZERO_SHOT_ENTRY_NAME = os.path.splitext(csv_name)[0]
    print(f"Uploaded CSV: {ZERO_SHOT_CSV_PATH}")
elif use_example_csv:
    ZERO_SHOT_CSV_PATH = TOY_REGRESSION_CSV
    if not os.path.exists(ZERO_SHOT_CSV_PATH):
        raise FileNotFoundError(f"Bundled toy CSV not found at {ZERO_SHOT_CSV_PATH}. Re-run setup (Cell 1).")
    print(f"Using bundled toy regression CSV: {ZERO_SHOT_CSV_PATH}")
else:
    raise ValueError("Enable upload_csv or use_example_csv to provide a regression dataset.")

zs_full_output_dir = os.path.join(zs_external_path, zs_output_dir)
os.makedirs(zs_full_output_dir, exist_ok=True)

print(f"Zero-shot output directory: {zs_full_output_dir}")
print(f"Entry name: {ZERO_SHOT_ENTRY_NAME}")

In [25]:
# @title Zero-shot — Add sequences to score
# @markdown Example sequences (GRB2 C-terminal region):
# @markdown - Seq 1 (wildtype): `TYVQALFDFDPQEDGELGFRRGDFIHVMDNSDPNWWKGACHGQTGMFPRNYVTPVNRNV`
# @markdown - Seq 2: `TYVQALFDFDPQEDGELGFRRGDFIEVMDNSDPNWWKGACHGQTGMFPRNYVTPVNRNV`
# @markdown - Seq 3: `TYVQALFDFDPQEDGELGFRRGDFIHVMDNSDPNWWKGACHGQTGMFPRNDVTPVNRNV`
# @markdown
# @markdown Note: All sequences must be the same length. Use `Add Sequence +` for more variants.

from IPython.display import display

ZERO_SHOT_SEQUENCES = [
    "TYVQALFDFDPQEDGELGFRRGDFIHVMDNSDPNWWKGACHGQTGMFPRNYVTPVNRNV",
    "TYVQALFDFDPQEDGELGFRRGDFIEVMDNSDPNWWKGACHGQTGMFPRNYVTPVNRNV",
    "TYVQALFDFDPQEDGELGFRRGDFIHVMDNSDPNWWKGACHGQTGMFPRNDVTPVNRNV",
]


class SequenceInputManager:
    def __init__(self, initial_sequences=None):
        self.sequences = []
        self.container = widgets.VBox()
        self.add_button = widgets.Button(description="Add Sequence +", icon="plus")
        self.add_button.on_click(self.add_field)
        if initial_sequences:
            for seq in initial_sequences:
                self.add_field(None, default_value=seq)
        else:
            self.add_field(None)

    def add_field(self, b, default_value=""):
        idx = len(self.sequences) + 1
        label = "Seq 1 (wildtype):" if idx == 1 else f"Seq {idx}:"
        text = widgets.Text(
            value=default_value,
            placeholder=f"Enter protein sequence {idx}...",
            layout=widgets.Layout(width='80%'),
        )
        box = widgets.HBox([widgets.Label(value=label, layout=widgets.Layout(width='120px')), text])
        self.sequences.append(text)
        self.container.children = tuple(list(self.container.children) + [box])

    def get_sequences(self):
        return [w.value.strip() for w in self.sequences if w.value.strip()]

    def display(self):
        display(widgets.VBox([self.container, self.add_button]))


zs_seq_manager = SequenceInputManager(initial_sequences=ZERO_SHOT_SEQUENCES)
zs_seq_manager.display()

In [ ]:
# @title Zero-shot — Run discovery and edge weights
# @markdown Discovers the zero-shot circuit from your CSV, scores the sequences above, then computes `virtual_weights.json` (may take a while on Colab).

sequences = zs_seq_manager.get_sequences()
if not sequences:
    raise ValueError("Add at least one sequence to score.")
lengths = [len(s) for s in sequences]
if len(set(lengths)) != 1:
    raise ValueError("All sequences must have the same length.")

cmd = [
    sys.executable, SCRIPT_PATH,
    "--type", "zero_shot",
    "--output_dir", zs_full_output_dir,
    "--entry_name", ZERO_SHOT_ENTRY_NAME,
    "--csv_path", ZERO_SHOT_CSV_PATH,
    "--max_nodes", "1000",
    "--step_size", "32",
    "--sequences", *sequences,
]

print("Starting zero-shot discovery...")
print(f"   CSV: {ZERO_SHOT_CSV_PATH}")
print(f"   Sequences: {len(sequences)}")
print(f"   Output: {zs_full_output_dir}/{ZERO_SHOT_ENTRY_NAME}.json\n")
exit_code = run_realtime(cmd)

if exit_code != 0:
    print("\nZero-shot discovery failed.")
else:
    ok = run_edge_weights(zs_full_output_dir, label="zero-shot")
    print("\n==========")
    print(f"Zero-shot results saved to: {zs_full_output_dir}")
    print("==========")
    print("Files generated:")
    for f in sorted(os.listdir(zs_full_output_dir)):
        suffix = "/" if os.path.isdir(os.path.join(zs_full_output_dir, f)) else ""
        print(f" - {f}{suffix}")
    if not ok:
        print("\nEdge-weight computation failed. See logs above.")